# Model 3: Revenue Forecasting (Prediksi Pendapatan)
**Tujuan:** Memproyeksikan pendapatan finansial (IDR) di masa depan berdasarkan tren performa views dan monetisasi.

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

## 2. Data Filtering
**Aturan:** Hanya eksekusi pada `monetization_rate = 1` atau `estimated_revenue_idr > 0`.

In [2]:
# Gunakan output dari Model 1 (yang sudah di-merge dengan fitur lain)
df = pd.read_csv('../../data/processed/forecast_results.csv')

# Kolom revenue aktual (target). Diasumsikan estimated_doubleclick_revenue_idr adalah revenue utamanya.
# Kita tambahkan validasi untuk memastikan tipe data benar.
if 'estimated_doubleclick_revenue_idr' not in df.columns:
    df['estimated_doubleclick_revenue_idr'] = 0

df_monetized = df[(df['monetization_rate'] == 1) | (df['estimated_doubleclick_revenue_idr'] > 0)].copy()

# Kita asumsikan revenue_per_view tidak null, jika null diisi median
if 'revenue_per_view' in df_monetized.columns:
    df_monetized['revenue_per_view'] = df_monetized['revenue_per_view'].fillna(df_monetized['revenue_per_view'].median())
else:
    df_monetized['revenue_per_view'] = 0.0

if 'ctr_impression_score' not in df_monetized.columns:
    df_monetized['ctr_impression_score'] = 0.0

print(f"Jumlah video monetisasi: {len(df_monetized)}")


Jumlah video monetisasi: 397


## 3. Modeling (Random Forest Regressor)
Fitur Input Wajib:
- Prediksi Views (Output Model 1)
- `revenue_per_view`
- `monetization_rate`
- `ctr_impression_score`

In [3]:
if not df_monetized.empty:
    features = ['predicted_views', 'revenue_per_view', 'monetization_rate', 'ctr_impression_score']
    target = 'estimated_doubleclick_revenue_idr'
    
    # Isi NaN dengan 0 untuk fitur training
    df_monetized[features] = df_monetized[features].fillna(0)
    df_monetized[target] = df_monetized[target].fillna(0)
    
    # Split Data (Simple 80-20 untuk testing/training evaluasi)
    split_idx = int(len(df_monetized) * 0.8)
    train = df_monetized.iloc[:split_idx]
    test  = df_monetized.iloc[split_idx:]
    
    X_train = train[features]
    y_train = train[target]
    X_test = test[features]
    y_test = test[target]
    
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    
    y_pred = rf_model.predict(X_test)


## 4. Evaluasi (WAPE)
WAPE (Weighted Absolute Percentage Error) agar model lebih fokus mengevaluasi akurasi pada video dengan *revenue* paling tinggi.

In [4]:
def calculate_wape(y_true, y_pred):
    sum_abs_error = np.sum(np.abs(y_true - y_pred))
    sum_abs_true = np.sum(np.abs(y_true))
    
    # Prevent division by zero
    if sum_abs_true == 0:
        return 0
    return sum_abs_error / sum_abs_true

if not df_monetized.empty:
    wape_score = calculate_wape(y_test, y_pred)
    print(f"WAPE Score: {wape_score:.4f} ({wape_score*100:.2f}%)")
    
    # Simpan prediksi revenue
    test['predicted_revenue'] = y_pred
    test[['video_id', 'predicted_views', 'predicted_revenue', 'estimated_doubleclick_revenue_idr']].head()


WAPE Score: 0.0000 (0.00%)
